# OpenEnv SmartGrid MarketSim - Colab Training Notebook
This notebook is Phase 3 training evidence for the hackathon submission. It supports Unsloth/HF TRL setup and environment-connected rollout collection.

In [ ]:
# Colab setup
!pip -q install unsloth trl transformers datasets accelerate matplotlib requests

## Configure Environment Endpoint
Set this to your deployed HF Space URL or localhost tunnel URL.

In [ ]:
import requests
import matplotlib.pyplot as plt

ENV_URL = "https://YOUR-HF-SPACE.hf.space"  # replace before run
TASK_ID = "default"

def api(path, method='GET', payload=None, params=None):
    fn = requests.get if method == 'GET' else requests.post
    r = fn(f"{ENV_URL}{path}", json=payload, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

## Collect Baseline and Adaptive Rollouts
This verifies environment-connected training/evaluation data generation.

In [ ]:
def run_rollout(policy='adaptive', personality='balanced', seed=42):
    out = api('/run-inference', method='POST', payload={
        'policy': policy,
        'personality': personality,
        'task_id': TASK_ID,
        'seed': seed,
    })
    rewards = [x['reward']['score'] for x in out['trajectory']]
    return out, rewards

baseline_curve = []
adaptive_curve = []
for i in range(10):
    _, b = run_rollout(policy='heuristic', personality='balanced', seed=1000+i)
    _, a = run_rollout(policy='adaptive', personality='opportunistic', seed=2000+i)
    baseline_curve.append(sum(b)/max(1,len(b)))
    adaptive_curve.append(sum(a)/max(1,len(a)))

print('baseline_avg=', sum(baseline_curve)/len(baseline_curve))
print('adaptive_avg=', sum(adaptive_curve)/len(adaptive_curve))

In [ ]:
plt.figure(figsize=(9,4))
plt.plot(baseline_curve, label='Heuristic baseline')
plt.plot(adaptive_curve, label='Adaptive policy')
plt.xlabel('Episode')
plt.ylabel('Average reward')
plt.title('Trained/Improved vs Baseline Comparison')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## Optional: Unsloth + TRL Fine-Tuning Scaffold
Use this when training a language policy model from collected trajectories.

In [ ]:
from datasets import Dataset

samples = []
for i in range(20):
    out, _ = run_rollout(policy='adaptive', personality='balanced', seed=3000+i)
    for t in out['trajectory']:
        samples.append({
            'prompt': str(t['info'].get('agent_private_views', {})),
            'response': str(t['action']),
            'reward': t['reward']['score'],
        })

ds = Dataset.from_list(samples)
print(ds)

## Export Judging Artifacts
Save generated plots and include them in the README + blog/video links.